In [ ]:
from misc import *
import constant

# upto 5 long track candidates isMuon
LTRACK5_FILE_PATH = "/dice/projects/LHCb/PbPb_MSCi/isMuon_Tr1_5_B2JpsiK_PbPb/LeadLead_B2JpsiKTuple_all.root"
DEFAULT_TREE_NAME = "B2JpsiKTuple/DecayTree"

df = load_root(LTRACK5_FILE_PATH, DEFAULT_TREE_NAME)
df = df[df["Kplus_isMuon"] == False]
df = df[(constant.JPSI_MASS-50 <= df["J_psi_1S_M"]) & (df["J_psi_1S_M"] <= constant.JPSI_MASS+50)]

: 

In [ ]:
# df, _ = filter_duplicates_by_performance(df, "Bplus_ENDVERTEX_CHI2", "min")
# df, _ = filter_duplicates_by_performance(df, "Kplus_PIDK", "max")

In [ ]:
import zfit
import zfit.plot
import matplotlib.pyplot as plt
import scipy

In [ ]:
bounds = (5000, 6000)

# Define the observable (entire range of the data)
obs = zfit.Space("m", limits=bounds)

data_np = df["Bplus_M"].to_numpy()
data_np = data_np[(data_np > bounds[0]) & (data_np < bounds[1])]

print(f"{len(data_np)=}")

data = zfit.Data.from_numpy(array=data_np, obs=obs)

mu = zfit.param.ConstantParameter("mu", constant.BPLUS_MASS) # fix gaussian mean
sigma = zfit.Parameter("sigma", 1)

lambda_ = zfit.Parameter("lambda", -0.001)

# calculate a crude upper limit for Nsig for parameter bounds
full_pm_20MeV_yield = len(df[(df["Bplus_M"] < constant.BPLUS_MASS + 20) & (df["Bplus_M"] > constant.BPLUS_MASS - 20)])

# Define the yields (Nsig and Nbkg)
Nsig = zfit.Parameter("Nsig", 0, 0, full_pm_20MeV_yield)  # Floating signal yield
Nbkg = zfit.Parameter("Nbkg", 300)  # Floating background yield

signal = zfit.pdf.Gauss(mu=mu, sigma=sigma, obs=obs).create_extended(Nsig)
background = zfit.pdf.Exponential(lam=lambda_, obs=obs).create_extended(Nbkg)

total = zfit.pdf.SumPDF([signal, background])

nll = zfit.loss.ExtendedUnbinnedNLL(model=total, data=data)

minimizer = zfit.minimize.Minuit()
minimum = minimizer.minimize(nll)
minimum.hesse()

print(minimum)

In [ ]:
test_points = np.linspace(0, 20, 21)
nll_vals = []
for point in test_points:
    Nsig.set_value(point)
    nll_vals.append(nll.value().numpy())
    
# Plot to visualize
plt.plot(test_points, nll_vals)
plt.xlabel(r'$N_{signal}$')
plt.ylabel(r'$-\log(\mathcal{L})$')

In [ ]:
def find_upper_limit(
    nll: zfit.loss.ExtendedUnbinnedNLL,
    Nsig: zfit.Parameter,
    minimum: zfit.result.FitResult,
    ndof=1,
    alpha=0.05, # CL = 1 - alpha
):
    def nll_from_Nsig(
        x: float, nll: zfit.loss.ExtendedUnbinnedNLL, Nsig: zfit.Parameter
    ):
        Nsig.set_value(x)
        return nll.value().numpy()

    nll_Nsig_hat = minimum.fmin  # i.e. minimum of nll
    Nsig_hat = float(minimum.params[Nsig]["value"])

    target_nll = scipy.stats.chi2.isf(alpha, ndof) / 2 + nll_Nsig_hat

    ul_Nsig: scipy.optimize.OptimizeResult = scipy.optimize.minimize_scalar(
        lambda x: abs(target_nll - nll_from_Nsig(x, nll, Nsig)),
        bounds=(Nsig_hat, float(Nsig.upper)),
    )

    return ul_Nsig.x


print(f"Upper limit for signal yield Nsig: {find_upper_limit(nll, Nsig, minimum)} at 95% CL")

print(f"{minimum.params}")

In [ ]:
bin_width = 15
fig1, ax1 = histogram_fig(
    [p := HistogramPlot(df["Bplus_M"], "Data", bins=bin_width)],
    False,
    unit_tex=r"\mathrm{MeV}/c^2",
    xlabel="Candidate Mass",
    xlim=[bounds[0]-50, bounds[1]+50],
    ylim=[0,50]
)

x = np.linspace(*bounds, 500)
ax1.plot(x, background.ext_pdf(x)*bin_width, label="Background", linewidth=1.8)
ax1.plot(x, signal.ext_pdf(x)*bin_width, label="Signal", linewidth=1.8)
ax1.plot(x, total.ext_pdf(x)*bin_width, label="Model", linewidth=1.8)
ax1.legend()

bin_width = 7
fig2, ax2 = histogram_fig(
    [p := HistogramPlot(df["Bplus_M"], "Data", bins=bin_width)],
    False,
    unit_tex=r"\mathrm{MeV}/c^2",
    xlabel="Candidate Mass",
    preset="Bplus Mass",
    ylim=[0,20]
)

x = np.linspace(5000, 5400, 500)
ax2.plot(x, background.ext_pdf(x)*bin_width, label="Background", linewidth=1.8)
ax2.plot(x, signal.ext_pdf(x)*bin_width, label="Signal", linewidth=1.8)
ax2.plot(x, total.ext_pdf(x)*bin_width, label="Model", linewidth=1.8)
ax2.legend()

In [ ]:
# from hepstats.hypotests.calculators import FrequentistCalculator, AsymptoticCalculator
# from hepstats.hypotests import UpperLimit
# from hepstats.hypotests.parameters import POI, POIarray

# calculator = AsymptoticCalculator(nll, zfit.minimize.Minuit())
# poinull = POIarray(Nsig, np.linspace(0, full_pm_20MeV_yield, 100))
# poialt = POI(Nsig, 1)

# ul = UpperLimit(calculator, poinull, poialt)
# ul.upperlimit(alpha=0.05)